In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("mabel.txt",header=None,error_bad_lines=False,encoding='utf8')

b'Skipping line 23: expected 2 fields, saw 6\nSkipping line 44: expected 2 fields, saw 3\n'


In [3]:
df=df.drop(0)
df.columns=['Date','Chat']
Message=df["Chat"].str.split("-",n=1,expand=True)
df["Time"]=Message[0]
Message1=Message[1].str.split(":",n=1,expand=True)
df["Name"]=Message1[0]
df["Chat"]=Message1[1]
df=df[["Date","Time","Name","Chat"]]
df

,Date,Time,Name,Chat
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K
4,05/12/19,1:42 pm,Mabel Infoziant,Ok
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?
7,05/12/19,1:43 pm,AR❤,Yes Mam
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working
9,05/12/19,1:43 pm,AR❤,Yeah mam
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50 entries, 1 to 50
Data columns (total 4 columns):
Date    50 non-null object
Time    50 non-null object
Name    50 non-null object
Chat    50 non-null object
dtypes: object(4)
memory usage: 2.0+ KB


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [6]:
tfidf=TfidfVectorizer(max_df=0.95,min_df=2,stop_words='english')
dtm=tfidf.fit_transform(df["Chat"])

In [7]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import Normalizer
# LDA Model
lda_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_topics = lda_model.fit_transform(dtm)

In [10]:
# Display top words per topic
def display_topics(model, feature_names, num_top_words):
    for idx, topic in enumerate(model.components_):
        print(f"\nTopic {idx+1}:")
        top_features = [feature_names[i] for i in topic.argsort()[:-num_top_words - 1:-1]]
        print(", ".join(top_features))
display_topics(lda_model, tfidf.get_feature_names(), 8)
# (Optional) Display dominant topic per chat
topic_assignments = lda_topics.argmax(axis=1)
topic_df = pd.DataFrame({'Chat': df["Chat"], 'Dominant_Topic': topic_assignments + 1})
print("\nChat Topic Assignments:\n", topic_df)


Topic 1:
ml, meeting, vignesh, project, start, soon, think, just

Topic 2:
ok, ramisha, send, abstracts, need, yes, number, details

Topic 3:
mam, tomorrow, yeah, hi, church, finiliaze, kk, sure

Chat Topic Assignments:
                                                  Chat  Dominant_Topic
1                      Hi this is Mabel we just spoke               1
2                               What’s your full name               1
3                                      Ramisha Rani K               2
4                                                  Ok               2
5                              ramisharanik@gmail.com               1
6                                      Your email Id?               1
7                                             Yes Mam               2
8      I will send 2 abstracts for u to start working               2
9                                            Yeah mam               3
10                   Give me the list that u have too               1
11   Sen